In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
import os

import pandas as pd
import mlflow

TABLE_NAME = "clean_users_churn"

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "churn_simonreise"
RUN_NAME = "autofeat" 
REGISTRY_MODEL_NAME = "churn_model_simonreise_catboost_basic"

In [23]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

In [13]:
import psycopg

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

df = df.drop(columns=["id", "end_date", "customer_id"])

df.head(2) 

,begin_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,2019-01-01,Month-to-month,Yes,Mailed check,49.95,587.45,DSL,Yes,No,No,No,No,No,Male,0,Yes,Yes,No,0
1,2016-09-01,Two year,No,Mailed check,20.65,835.15,Fiber optic,No,No,No,No,No,No,Female,0,No,No,No,0


In [14]:
from sklearn.model_selection import train_test_split

split_column = "begin_date"
test_size = 0.2

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["target"]),
    df["target"],
    test_size=test_size,
    shuffle=False,
) 

In [15]:
from autofeat import AutoFeatClassifier

In [20]:
cat_features = [
    'paperless_billing',
    'payment_method',
    'internet_service',
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies',
    'gender',
    'senior_citizen',
    'partner',
    'dependents',
    'multiple_lines',
]
num_features = ["monthly_charges", "total_charges"]

features = cat_features + num_features

transformations = ["1/", "log", "abs", "sqrt"]

afc = AutoFeatClassifier(
    transformations=transformations,
    categorical_cols=cat_features,
    feateng_cols=num_features,
    feateng_steps=1,
    n_jobs=-1,
)

X_train_features = afc.fit_transform(X_train[features], y_train)
X_test_features = afc.transform(X_test[features])

/home/mle-user/mle_projects/mle-mlflow/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [24]:
artifact_path = "afc"
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    afc_info = mlflow.sklearn.log_model(afc, artifact_path=artifact_path)
    print(run_id)

2026-03-08 08:58:41,165 INFO: Found credentials in environment variables.


e737a1d0e9a64f4aaba2011d32b58edf


In [27]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(auto_class_weights='Balanced')
model.fit(X_train_features, y_train)

Learning rate set to 0.021523
0:	learn: 0.6845151	total: 67ms	remaining: 1m 6s
1:	learn: 0.6757445	total: 70.7ms	remaining: 35.3s
2:	learn: 0.6682112	total: 74.2ms	remaining: 24.7s
3:	learn: 0.6620850	total: 77.6ms	remaining: 19.3s
4:	learn: 0.6546535	total: 81ms	remaining: 16.1s
5:	learn: 0.6490428	total: 84.4ms	remaining: 14s
6:	learn: 0.6441465	total: 87.7ms	remaining: 12.4s
7:	learn: 0.6385072	total: 91ms	remaining: 11.3s
8:	learn: 0.6328958	total: 94.3ms	remaining: 10.4s
9:	learn: 0.6268481	total: 97.7ms	remaining: 9.67s
10:	learn: 0.6223590	total: 101ms	remaining: 9.09s
11:	learn: 0.6174634	total: 104ms	remaining: 8.6s
12:	learn: 0.6122006	total: 108ms	remaining: 8.19s
13:	learn: 0.6083541	total: 111ms	remaining: 7.84s
14:	learn: 0.6036205	total: 115ms	remaining: 7.54s
15:	learn: 0.6000075	total: 118ms	remaining: 7.27s
16:	learn: 0.5955926	total: 122ms	remaining: 7.04s
17:	learn: 0.5929284	total: 126ms	remaining: 6.85s
18:	learn: 0.5882041	total: 129ms	remaining: 6.66s
19:	learn:

In [28]:
prediction = model.predict(X_test_features)
probas = model.predict_proba(X_test_features)

In [29]:
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss
# импортируйте необходимые вам модули

# заведите словарь со всеми метриками
metrics = {}

# посчитайте метрики из модуля sklearn.metrics
# err_1 — ошибка первого рода
# err_2 — ошибка второго рода
_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize="all").ravel()
auc = roc_auc_score(y_test, probas[:, 1])
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, prediction)

# запишите значения метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [30]:
pip_requirements = "./requirements.txt"
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]
metadata = {"model_type": "monthly"}

/home/mle-user/mle_projects/mle-mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None


In [31]:
RUN_NAME = "model_feat_generation"

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    # ваш код здесь
    model_info = mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="models",
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
        metadata=metadata,
        registered_model_name=REGISTRY_MODEL_NAME,
        await_registration_for=60,
    )
    mlflow.log_metrics(metrics)

Registered model 'churn_model_simonreise_catboost_basic' already exists. Creating a new version of this model...
2026/03/08 09:10:28 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model version to finish creation. Model name: churn_model_simonreise_catboost_basic, version 5
Created version '5' of model 'churn_model_simonreise_catboost_basic'.
